- 목표 : visdom 사용법을 익히고 MNIST-CNN loss graph까지 적용

# visdom example

** python -m visdom.server ** 키고 해야 함. 

In [2]:
import torch
import torch.nn as nn

import torchvision
import torchvision.datasets as dsets

## visdom import 하기
import visdom
vis = visdom.Visdom()

Setting up a new session...


In [3]:
# text
vis.text("Hello World!", env="main")

'window_3eeee43ac6e572'

# images
a = torch.randn(3, 200, 200)
vis.image(a)

vis.images(torch.Tensor(3,3,28,28))

In [4]:
# example (using MNIST and CIFAR10)

MNIST = dsets.MNIST(root='./MNIST_data', train=True, transform=torchvision.transforms.ToTensor(), download=True)
cifar10 = dsets.CIFAR10(root='./cifar10', train=True, transform=torchvision.transforms.ToTensor(), download=True)

In [5]:
# CIFAR10

data = cifar10.__getitem__(0)
print(data[0].shape)
vis.images(data[0], env="main")

# MNIST

data = MNIST.__getitem__(0)
print(data[0].shape)
vis.images(data[0], env="main")

torch.Size([3, 32, 32])
torch.Size([1, 28, 28])


'window_3eeee43f5050c0'

In [6]:
# check dataset
data_loader = torch.utils.data.DataLoader(dataset=MNIST, batch_size=32, shuffle=False)

for num, value in enumerate(data_loader):
    value = value[0]
    print(value.shape)
    vis.images(value)
    break

torch.Size([32, 1, 28, 28])


In [7]:
# 창 모두 닫기
vis.close(env="main")

''

In [8]:
# line plot
Y_data = torch.randn(5)
plt = vis.line(Y_data)

X_data = torch.Tensor([1, 2, 3, 4, 5])
plt = vis.line(Y=Y_data, X=X_data)

In [9]:
# line update
Y_append = torch.randn(1)
X_append = torch.Tensor([6])

vis.line(Y=Y_append, X=X_append, win=plt, update='append')

'window_3eeee443389628'

In [10]:
# multiple line on single windows
num = torch.Tensor(list(range(0, 10)))
print(num.shape)
num = num.view(-1, 1)
print(num.shape)
num = torch.cat((num, num), dim=1)
print(num.shape)

plt = vis.line(Y=torch.randn(10, 2), X=num)

torch.Size([10])
torch.Size([10, 1])
torch.Size([10, 2])


In [11]:
# line info
# showlegend : legend 이름 보여주기. 
plt = vis.line(Y=Y_data, X=X_data, opts = dict(title='Test', showlegend=True))
plt = vis.line(Y=Y_data, X=X_data, opts=dict(title='Test', legend=['1번'], showlegend=True))
plt = vis.line(Y=torch.randn(10, 2), X = num, opts=dict(title='Test', legend=['1번', '2번'], showlegend=True))

In [ ]:
# make funtion for update line : 그래프에 새로운 데이터 추가 역할
def loss_tracker(loss_plot, loss_value, num):
    '''num, loss_value, are Tensor'''
    vis.line(X=num,
             Y=loss_value,
             win=loss_plot,
             update='append')

plt = vis.line(Y=torch.Tensor(1).zero_())

for i in range(500):
    loss = torch.randn(1) + i
    loss_tracker(plt, loss, torch.Tensor([i]))

In [13]:
# close the window
vis.close(env="main")

''

# MNIST-CNN with Visdom

In [14]:
import torch
import torch.nn as nn
import torchvision.datasets as dsets
import torchvision.transforms as transforms

import torch.nn.init

### import visdom

In [15]:
import visdom
vis = visdom.Visdom()
vis.close(env="main")

Setting up a new session...


''

### define loss_tracker

In [16]:
def loss_tracker(loss_plot, loss_value, num):
    '''num, loss_value, are Tensor
       line의 win 인자는 함수에서 전달받은 ID가 할당되어 정확히 해당 ID 창에 로스값 업뎃'''
    vis.line(X=num,
             Y=loss_value,
             win=loss_plot,
             update='append')

In [17]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

In [18]:
learning_rate = 0.001
training_epochs = 15
batch_size = 32

In [20]:
# MNIST dataset
mnist_train = dsets.MNIST(root='MNIST_data/', 
                          train=True,
                          transform=transforms.ToTensor(),
                          download=True)

mnist_test = dsets.MNIST(root='MNIST_data/',
                         train=False,
                         transform=transforms.ToTensor(),
                         download=True)

In [21]:
data_loader = torch.utils.data.DataLoader(dataset=mnist_train,
                                          batch_size=batch_size,
                                          shuffle=True,
                                          drop_last=True)

In [25]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.layer3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc1 = nn.Linear(3 * 3 * 128, 625)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(625, 10, bias=True)
        torch.nn.init.xavier_uniform_(self.fc1.weight)
        torch.nn.init.xavier_uniform_(self.fc2.weight)
    
    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)

        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        return out

In [26]:
model = CNN().to(device)

value = (torch.Tensor(1, 1, 28, 28)).to(device)
print((model(value)).shape)

torch.Size([1, 10])


In [28]:
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

### make plot

In [27]:
loss_plt = vis.line(Y=torch.Tensor(1).zero_(), opts=dict(title='loss_tracker', legend=['loss'], showlegend=True))

### train with loss_tracker

In [ ]:
total_batch = len(data_loader)

for epoch in range(training_epochs):
    avg_cost = 0

    for X, Y in data_loader:
        X = X.to(device)
        Y = Y.to(device)
        
        optimizer.zero_grad()
        hypothesis = model(X)

        cost = criterion(hypothesis, Y)
        cost.backward()
        optimizer.step()

        avg_cost += cost / total_batch
    
    print('[Epoch:{}] cost = {}'.format(epoch+1, avg_cost))
    loss_tracker(loss_plt, torch.Tensor([avg_cost]), torch.Tensor([epoch]))
print('Learning finished!')

[Epoch:1] cost = 0.12299329787492752
[Epoch:2] cost = 0.03891078010201454
[Epoch:3] cost = 0.027246683835983276
[Epoch:4] cost = 0.020472975447773933
[Epoch:5] cost = 0.0176613200455904


In [ ]:
with torch.no_grad():
    X_test = mnist_test.test_data.view(len(mnist_test), 1, 28, 28).float().to(device)
    Y_test = mnist_test.test_labels.to(device)

    prediction = model(X_test)
    correct_prediction = torch.argmax(prediction, 1) == Y_test
    accuracy = correct_prediction.float().mean()
    print('Accuracy:', accuracy.item())